<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/notebook-02-boundary-token/notebooks/02_tokenizer_training_and_corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 02 — Tokenizer Training & Corpus Construction

This notebook begins from the verified data contract established in **Notebook 01 — Data Preparation & Corpus Audit**. It independently reloads the immutable WikiText-103 revision, applies the same locked normalization and article-reconstruction logic through `src/data.py`, trains the project tokenizer from scratch, validates it, records its checksum, and constructs the exact 20,000,000-token model-training corpus.

Notebook 01 is a completed audit artifact. Notebook 02 must not depend on Notebook 01's in-memory state.

### Pipeline boundary

```text
Notebook 01: raw WikiText → verified normalized articles
src/data.py: canonical reusable loading/normalization/reconstruction logic
Notebook 02: normalized articles → tokenizer → exact 20M-token corpus
Notebook 03: tokenizer/corpus → Transformer architecture
```


### Locked inputs and constraints

- Dataset: `Salesforce/wikitext`, `wikitext-103-raw-v1`
- Immutable Hub revision: `b08601e04326c79dfdd32d625aee71d232d685c3`
- Tokenizer: byte-level BPE trained from scratch
- Vocabulary size: 16,384 total tokens, including registered special tokens
- Tokenizer-training text: full normalized official training split only
- Model-training corpus: exactly 20,000,000 tokenizer-produced tokens
- Sampling seed: 42
- Validation remains development-visible; test remains untouched until final evaluation
- Article boundaries and normalization must reproduce Notebook 01's verified 28,472 training documents and 60 validation documents before tokenizer work proceeds

Canonical references: `docs/PROJECT_CONTEXT.md` and `docs/DECISION_REGISTER.md`.

# Chunk 1 — Reproduce the audited corpus from shared source code

Before tokenizer design begins, this chunk proves that Notebook 02 can independently reproduce Notebook 01's audited corpus. The verified normalization and article-reconstruction implementation has been extracted into `src/data.py` without refactoring the core logic.

The notebook pins the **source-code revision** containing that extraction. This matters because the Hub dataset revision alone fixes the upstream text, while the source-code revision fixes the exact transformation applied to it.

## 1. Load the canonical data pipeline at a fixed source revision

A Colab notebook opened from GitHub does not automatically make the repository's `src/` package importable. We therefore clone the project repository, check out the exact commit that introduced the verified extraction, and add the repository root to Python's import path.

This is intentionally a source-code pin, not a dependency install. The goal is for a fresh runtime to reconstruct the same data pipeline without relying on Notebook 01 or on whatever happens to be at the tip of `main` later.

In [1]:
%pip install -q datasets


In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/traderjohnd/foundation-model-from-scratch.git"
REPO_DIR = Path("/content/foundation-model-from-scratch")
DATA_PIPELINE_REVISION = "7d300f14c812d9a1caf36aa9ec0568bee5b0f275"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", DATA_PIPELINE_REVISION], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Loaded project source revision: {DATA_PIPELINE_REVISION}")


Loaded project source revision: 7d300f14c812d9a1caf36aa9ec0568bee5b0f275


In [3]:
import pandas as pd

from src.data import (
    AUDIT_SPLITS,
    DATASET_CONFIG,
    DATASET_ID,
    DATASET_REVISION,
    EXPECTED_ARTICLE_COUNTS,
    load_pinned_wikitext,
    normalize_development_splits,
    reconstruct_development_articles,
    run_normalization_self_test,
)

pd.Series({
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "source_revision": DATA_PIPELINE_REVISION,
    "development_splits": AUDIT_SPLITS,
})


dataset                                    Salesforce/wikitext
config                                     wikitext-103-raw-v1
dataset_revision      b08601e04326c79dfdd32d625aee71d232d685c3
source_revision       7d300f14c812d9a1caf36aa9ec0568bee5b0f275
development_splits                         (train, validation)
dtype: object

## 2. Re-run the locked normalization regression tests

Notebook 01 established 17 explicit normalization cases. Running the same cases through `src/data.py` checks that extraction did not silently change the transformation contract.

In [4]:
normalization_test_results = run_normalization_self_test()
normalization_tests = pd.DataFrame(normalization_test_results)

assert len(normalization_tests) == 17
assert normalization_tests["passed"].all()
print("✓ 17/17 normalization regression tests passed.")
normalization_tests


✓ 17/17 normalization regression tests passed.


## 3. Reload the immutable WikiText-103 revision

The module loads the same pinned Hub commit used by Notebook 01 and hard-checks the official row counts. Loading the dataset does **not** authorize development-time inspection of test examples; only `train` and `validation` are transformed below.

In [5]:
raw_dataset = load_pinned_wikitext()

split_rows = pd.Series(
    {split_name: raw_dataset[split_name].num_rows for split_name in raw_dataset},
    name="rows",
)
split_rows


test             4358
train         1801350
validation       3760
Name: rows, dtype: int64

## 4. Normalize development-visible splits only

The same locked function is applied to the official training and validation splits. The test split is deliberately not normalized or inspected during development.

In [6]:
normalized_development = normalize_development_splits(raw_dataset)

assert set(normalized_development) == {"train", "validation"}
print("✓ Normalized only train and validation.")


✓ Normalized only train and validation.


## 5. Reconstruct articles and enforce the audit contract

Article starts are detected from the **raw** rows using the locked level-1-heading-plus-blank-neighbors rule, while the stored article text comes from the normalized rows. This is the same distinction that resolved the boundary-count discrepancies in Notebook 01.

The hard assertions below are the handoff gate. Tokenizer work does not proceed unless the shared module independently reproduces **28,472 training documents and 60 validation documents**.

In [7]:
articles_by_split = reconstruct_development_articles(
    raw_dataset,
    normalized_development,
)

article_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles": len(articles),
            "expected": EXPECTED_ARTICLE_COUNTS[split_name],
            "characters": sum(len(article["text"]) for article in articles),
            "first_article_id": articles[0]["article_id"],
            "last_article_id": articles[-1]["article_id"],
        }
        for split_name, articles in articles_by_split.items()
    ]
).set_index("split")

assert article_summary.loc["train", "articles"] == 28_472
assert article_summary.loc["validation", "articles"] == 60
print("✓ Shared pipeline reproduced the audited article counts.")
article_summary


✓ Shared pipeline reproduced the audited article counts.


            articles  expected  characters                first_article_id  \
split                                                                        
train          28472     28472   519078250       train:article-00000:row-1   
validation        60        60     1102768  validation:article-00000:row-1   

                              last_article_id  
split                                          
train         train:article-28471:row-1801326  
validation  validation:article-00059:row-3714  

### Chunk 1 checkpoint — passed

A fresh runtime reloaded the pinned upstream corpus, executed the canonical shared preprocessing implementation, reran all 17 normalization tests, and reproduced the audited **28,472 training / 60 validation** article counts. Notebook 02 can therefore proceed to tokenizer design without depending on Notebook 01 kernel state.

# Chunk 2 — Lock the document-boundary / EOS special-token contract

Before BPE training begins, we need to decide which symbols are **structural** rather than learned from ordinary text. The project will use **`<|endoftext|>`** as the single registered special token. It serves as the document-boundary / end-of-sequence marker that will later be appended explicitly after each complete article.

This is a vocabulary contract, not tokenizer training. No BPE merges are learned in this chunk.

## 6. Define the structural vocabulary contract

The total vocabulary remains **16,384 IDs**, and the special token is reserved **inside** that total. We register no PAD, BOS, or UNK token.

- **No PAD:** the training pipeline will use fixed-length token sequences rather than padded variable-length examples.
- **No BOS:** the project does not need a separate beginning-of-sequence marker; document separation is handled by the boundary/EOS token.
- **No UNK:** byte-level BPE is designed to retain byte coverage, so an unknown-token fallback is unnecessary. Byte coverage will be explicitly validated after tokenizer training.
- **One boundary/EOS token:** `<|endoftext|>` is inserted by corpus construction, not automatically by ordinary text encoding.

Because the byte-level alphabet contains 256 byte symbols, a final 16,384-token vocabulary that reaches its requested size would contain 1 special token, 256 byte-alphabet tokens, and up to 16,127 merge-created vocabulary entries. We will verify the actual trained vocabulary rather than assuming the trainer reaches this exact composition.

In [8]:
VOCAB_SIZE = 16_384
DOC_BOUNDARY_TOKEN = "<|endoftext|>"
SPECIAL_TOKENS = [DOC_BOUNDARY_TOKEN]
BYTE_ALPHABET_SIZE = 256

assert len(SPECIAL_TOKENS) == 1
assert VOCAB_SIZE > BYTE_ALPHABET_SIZE + len(SPECIAL_TOKENS)

vocab_contract = pd.Series({
    "total_vocab_size": VOCAB_SIZE,
    "registered_special_tokens": len(SPECIAL_TOKENS),
    "document_boundary_token": DOC_BOUNDARY_TOKEN,
    "byte_alphabet_size": BYTE_ALPHABET_SIZE,
    "non_special_vocab_slots": VOCAB_SIZE - len(SPECIAL_TOKENS),
    "max_merge_created_slots_if_full": (
        VOCAB_SIZE - len(SPECIAL_TOKENS) - BYTE_ALPHABET_SIZE
    ),
})

vocab_contract


total_vocab_size                           16384
registered_special_tokens                      1
document_boundary_token            <|endoftext|>
byte_alphabet_size                           256
non_special_vocab_slots                    16383
max_merge_created_slots_if_full            16127
dtype: object

## 7. Prove the boundary-token string does not occur naturally

A registered special token must have an unambiguous structural meaning. If the literal string `<|endoftext|>` already appeared inside an article, that natural text could be confused with the boundary marker once the tokenizer treats the string as a special symbol.

We therefore search only the development-visible reconstructed `train` and `validation` articles. The test split remains uninspected.

In [9]:
boundary_collision_counts = {}

for split_name in AUDIT_SPLITS:
    literal_matches = sum(
        DOC_BOUNDARY_TOKEN in article["text"]
        for article in articles_by_split[split_name]
    )
    boundary_collision_counts[split_name] = literal_matches
    assert literal_matches == 0, (
        f"{DOC_BOUNDARY_TOKEN!r} occurs literally in {split_name}: "
        f"{literal_matches} articles"
    )

print("✓ Boundary-token literal collision check passed.")
pd.Series(boundary_collision_counts, name="articles_with_literal_boundary_token")


✓ Boundary-token literal collision check passed.


train         0
validation    0
Name: articles_with_literal_boundary_token, dtype: int64

## 8. Check whether literal `<unk>` appears as ordinary corpus text

We do **not** register `<unk>` as a tokenizer special token. Byte-level BPE does not require an unknown-token fallback when byte coverage is preserved, and turning a naturally occurring literal `<unk>` string into a control symbol would change its semantics if such text were present.

Rather than assume whether the audited corpus contains that literal string, we measure it in development-visible reconstructed articles. This is an observation, not a hard-coded assertion; the observed counts are recorded after execution.

In [10]:
LITERAL_UNK = "<unk>"

unk_inventory = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles_with_literal_unk": sum(
                LITERAL_UNK in article["text"]
                for article in articles_by_split[split_name]
            ),
            "literal_unk_occurrences": sum(
                article["text"].count(LITERAL_UNK)
                for article in articles_by_split[split_name]
            ),
        }
        for split_name in AUDIT_SPLITS
    ]
).set_index("split")

unk_inventory


            articles_with_literal_unk  literal_unk_occurrences
split                                                         
train                               0                        0
validation                          0                        0

### Pause here

This chunk locks the special-token contract and checks the corpus against it. **Do not train the tokenizer yet.**

Execution gate for the next chunk:
- the boundary-token collision count must be zero for both train and validation;
- the observed literal `<unk>` counts should be recorded without turning `<unk>` into a special token; and
- the 16,384-token vocabulary accounting must remain explicit.

**Next reviewed chunk:** configure the actual byte-level BPE tokenizer and verify the special-token ID / byte-level mechanics before full tokenizer training.